In [0]:
import hashlib
import json
import re
import unicodedata

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")


def normalize_text(value):
    if not isinstance(value, str):
        return ""
    return " ".join(value.split())


@F.udf("string")
def geocode_file_name(stadium_name, stadium_location, country):
    stadium_name = normalize_text(stadium_name)
    stadium_location = normalize_text(stadium_location)
    country = normalize_text(country)

    if not stadium_name:
        return None

    location_parts = [
        normalize_text(part)
        for part in stadium_location.split(",")
        if normalize_text(part)
    ]
    city_hint = location_parts[-1] if len(location_parts) > 1 else ""

    components = [stadium_name, city_hint]
    if country.casefold() != "europe":
        components.append(country)

    query_parts = []
    seen = set()

    for component in components:
        key = component.casefold()
        if component and key not in seen:
            query_parts.append(component)
            seen.add(key)

    query = ", ".join(query_parts)

    venue_identity = json.dumps(
        {
            "stadium_location": stadium_location.casefold(),
            "stadium_name": stadium_name.casefold(),
        },
        ensure_ascii=True,
        sort_keys=True,
        separators=(",", ":"),
    )

    geocode_identity = json.dumps(
        {
            "query": query.casefold(),
            "venue": venue_identity,
        },
        ensure_ascii=True,
        sort_keys=True,
        separators=(",", ":"),
    )

    query_hash = hashlib.sha256(
        geocode_identity.encode("utf-8")
    ).hexdigest()[:12]

    ascii_name = (
        unicodedata.normalize("NFKD", stadium_name)
        .encode("ascii", "ignore")
        .decode()
    )
    slug = re.sub(
        r"[^a-z0-9]+",
        "-",
        ascii_name.casefold(),
    ).strip("-")[:48] or "venue"

    return f"venue_{slug}_{query_hash}.json"


@F.udf("string")
def stable_venue_id(stadium_name, stadium_location, osm_type, osm_id):
    stadium_name = normalize_text(stadium_name)
    stadium_location = normalize_text(stadium_location)

    if not stadium_name:
        return None

    if osm_type and osm_id is not None:
        identity = f"nominatim:osm:{osm_type.casefold()}:{osm_id}"
    else:
        identity = json.dumps(
            {
                "stadium_location": stadium_location.casefold(),
                "stadium_name": stadium_name.casefold(),
            },
            ensure_ascii=True,
            sort_keys=True,
            separators=(",", ":"),
        )

    digest = hashlib.sha256(
        identity.encode("utf-8")
    ).hexdigest()[:16].upper()

    return f"VENUE-{digest}"

In [0]:
geocode_schema = T.ArrayType(
    T.StructType([
        T.StructField("osm_type", T.StringType()),
        T.StructField("osm_id", T.LongType()),
    ])
)

geocodes = (
    spark.table("clubdata.bronze.geocoding_raw")
    .select(
        F.col("source_file").alias("geocode_file"),
        F.get(
            F.from_json("raw_payload", geocode_schema),
            0,
        ).alias("geocode"),
    )
    .select(
        "geocode_file",
        F.col("geocode.osm_type").alias("osm_type"),
        F.col("geocode.osm_id").alias("osm_id"),
    )
)

display(geocodes)

In [0]:
raw_matches = spark.table("clubdata.bronze.matches_raw")

candidates = (
    raw_matches
    .select(
        "source_match_id",
        "record_hash",
        "payload",
        F.col("payload.date_unix").cast("long").alias("kickoff_epoch"),
        F.col("payload.league.competition_name").alias("competition"),
        F.col("payload.league.country").alias("league_country"),
        F.col("payload.season.year").cast("long").alias("season"),
        F.col("payload.home_team.team_id")
            .cast("long")
            .alias("home_team_id"),
        F.col("payload.home_team.team_name").alias("home_team_name"),
        F.col("payload.away_team.team_id")
            .cast("long")
            .alias("away_team_id"),
        F.col("payload.away_team.team_name").alias("away_team_name"),
        F.col("payload.score.home").cast("long").alias("home_score"),
        F.col("payload.score.away").cast("long").alias("away_score"),
        F.col("payload.status").alias("status"),
        F.col("payload.venue.stadium_name").alias("venue_name_raw"),
        F.col("payload.venue.stadium_location")
            .alias("venue_location_raw"),
        F.length(F.to_json("payload")).alias("completeness_score"),
    )
    .withColumn(
        "geocode_file",
        geocode_file_name(
            "venue_name_raw",
            "venue_location_raw",
            "league_country",
        ),
    )
    .join(geocodes, on="geocode_file", how="left")
    .withColumn(
        "venue_id",
        stable_venue_id(
            "venue_name_raw",
            "venue_location_raw",
            "osm_type",
            "osm_id",
        ),
    )
)

logical_match = Window.partitionBy(
    "kickoff_epoch",
    "competition",
    "season",
    "home_team_id",
    "away_team_id",
).orderBy(
    F.desc("completeness_score"),
    F.desc(F.when(F.col("status") == "complete", 1).otherwise(0)),
    F.desc("source_match_id"),
)

silver_matches = (
    candidates
    .withColumn("deduplication_rank", F.row_number().over(logical_match))
    .filter(F.col("deduplication_rank") == 1)
    .select(
        F.col("source_match_id").alias("match_id"),
        F.to_timestamp(F.from_unixtime("kickoff_epoch")).alias("kickoff_at"),
        "competition",
        "season",
        "home_team_id",
        "home_team_name",
        "away_team_id",
        "away_team_name",
        "home_score",
        "away_score",
        "status",
        "venue_id",
        F.when(
            F.trim("venue_name_raw") == "",
            None,
        ).otherwise(F.trim("venue_name_raw")).alias("venue_name"),
        F.when(
            F.trim("venue_location_raw") == "",
            None,
        ).otherwise(F.trim("venue_location_raw")).alias(
            "venue_location_raw"
        ),
        F.lit(None).cast("long").alias("attendance"),
        F.lit("FootballData").alias("source"),
    )
)

In [0]:
(
    silver_matches.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("clubdata.silver.matches")
)

print("Opprettet clubdata.silver.matches")

In [0]:
%sql

SELECT
    COUNT(*) AS silver_rows,
    COUNT(DISTINCT match_id) AS distinct_match_ids,
    COUNT_IF(
        match_id IS NULL
        OR kickoff_at IS NULL
        OR home_team_name IS NULL
        OR away_team_name IS NULL
    ) AS invalid_required_rows,
    COUNT_IF(venue_id IS NOT NULL) AS matches_with_venue
FROM clubdata.silver.matches;

In [0]:
full_geocode_schema = T.ArrayType(
    T.StructType([
        T.StructField("osm_type", T.StringType()),
        T.StructField("osm_id", T.LongType()),
        T.StructField("lat", T.StringType()),
        T.StructField("lon", T.StringType()),
        T.StructField("display_name", T.StringType()),
        T.StructField(
            "address",
            T.StructType([
                T.StructField("country", T.StringType()),
            ]),
        ),
    ])
)

full_geocodes = (
    spark.table("clubdata.bronze.geocoding_raw")
    .withColumn(
        "results",
        F.from_json("raw_payload", full_geocode_schema),
    )
    .select(
        F.col("source_file").alias("geocode_file"),
        F.get("results", 0).alias("geocode"),
    )
    .select(
        "geocode_file",
        F.col("geocode.osm_type").alias("osm_type"),
        F.col("geocode.osm_id").alias("osm_id"),
        F.col("geocode.lat").cast("double").alias("latitude"),
        F.col("geocode.lon").cast("double").alias("longitude"),
        F.col("geocode.display_name").alias("geocoding_display_name"),
        F.col("geocode.address.country").alias("country"),
    )
)

In [0]:
@F.udf("string")
def build_geocode_query(stadium_name, stadium_location, country):
    stadium_name = normalize_text(stadium_name)
    stadium_location = normalize_text(stadium_location)
    country = normalize_text(country)

    if not stadium_name:
        return None

    location_parts = [
        normalize_text(part)
        for part in stadium_location.split(",")
        if normalize_text(part)
    ]
    city_hint = location_parts[-1] if len(location_parts) > 1 else ""

    components = [stadium_name, city_hint]
    if country.casefold() != "europe":
        components.append(country)

    query_parts = []
    seen = set()

    for component in components:
        key = component.casefold()
        if component and key not in seen:
            query_parts.append(component)
            seen.add(key)

    return ", ".join(query_parts)

In [0]:
match_countries = (
    spark.table("clubdata.bronze.matches_raw")
    .select(
        F.col("source_match_id").alias("match_id"),
        F.col("payload.league.country").alias("league_country"),
    )
)

venue_candidates = (
    spark.table("clubdata.silver.matches")
    .filter(F.col("venue_id").isNotNull())
    .join(match_countries, on="match_id", how="left")
    .withColumn(
        "geocode_file",
        geocode_file_name(
            "venue_name",
            "venue_location_raw",
            "league_country",
        ),
    )
    .withColumn(
        "geocoding_query",
        build_geocode_query(
            "venue_name",
            "venue_location_raw",
            "league_country",
        ),
    )
    .join(full_geocodes, on="geocode_file", how="left")
)

In [0]:
representative_venue = Window.partitionBy("venue_id").orderBy(
    F.desc(F.when(F.col("osm_type").isNotNull(), 1).otherwise(0)),
    F.desc(
        F.when(F.col("venue_location_raw").isNotNull(), 1).otherwise(0)
    ),
    F.desc(
        F.length(F.coalesce("venue_location_raw", F.lit("")))
    ),
    F.asc(
        F.when(F.instr("venue_name", "(") > 0, 1).otherwise(0)
    ),
    F.asc(F.length("venue_name")),
    F.asc(F.lower("venue_name")),
)

silver_venues = (
    venue_candidates
    .withColumn(
        "representative_rank",
        F.row_number().over(representative_venue),
    )
    .filter(F.col("representative_rank") == 1)
    .select(
        "venue_id",
        F.col("venue_name").alias("stadium_name"),
        "venue_location_raw",
        "latitude",
        "longitude",
        F.lit("Nominatim").alias("geocoding_provider"),
        "geocoding_query",
        "geocoding_display_name",
        F.lit(None).cast("double").alias("geocoding_confidence"),
        "country",
    )
)

In [0]:
(
    silver_venues.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("clubdata.silver.venues")
)

print("Opprettet clubdata.silver.venues")

